# Map of the Universe

Two maps of the large-scale structure of the Universe from the SDSS spectroscopic surveys:

1. **Equatorial slice:** galaxies of the SDSS Main Galaxy Sample in a thin strip around the celestial equator, colored by their rest-frame color.
2. **Multi-tracer wedge:** the same strip out to redshift $3.5$, combining the galaxy and quasar samples of the SDSS, BOSS and eBOSS surveys.

Database ideas in this notebook:

- selecting a survey sample by its target bit with a bitmask function;
- a `LEFT JOIN` that keeps rows missing from the joined table;
- counting rows on the server before downloading them;
- building one query per row of a table of sample definitions.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from SciServer import CasJobs

import settings

# The queries in this notebook need no SciServer login. Silence the
# warning SciScript gives about the missing login token.
warnings.filterwarnings("ignore", message="In Authentication.getToken")

In [ ]:
def draw_map_axes(ax, z_max, z_step):
    """
    Axes of a polar map of right ascension and comoving distance.

    Ticks on the rim mark right ascension. A radial axis runs from the
    center straight down to z_max, with comoving distance ticks on its
    left side and redshift ticks every z_step, labelled every second
    one, on its right side.
    """
    redshifts = np.linspace(0, z_max, 1000)
    distances = settings.COSMOLOGY.comoving_distance(redshifts).value
    ax.set_rlim(0, distances[-1])
    ax.set_rticks([])
    ax.grid(False)
    # The "top" ticks of the angle axis of polar axes sit on the rim.
    ax.tick_params(axis="x", top=True)

    # Zero-width axes from the center of the map down to the rim.
    radial_axis = ax.inset_axes([0.5, 0, 0, 0.5])
    radial_axis.set_facecolor("none")
    radial_axis.spines[["top", "bottom", "right"]].set_visible(False)
    radial_axis.set_xticks([])
    radial_axis.set_ylim(distances[-1], 0)
    radial_axis.minorticks_off()
    radial_axis.yaxis.get_major_locator().set_params(prune="lower")
    radial_axis.yaxis.tick_left()
    radial_axis.tick_params(direction="out")
    radial_axis.set_ylabel("comoving distance (Mpc)")

    # Conversions from comoving distance to redshift and back.
    conversions = (
        lambda d: np.interp(d, distances, redshifts),
        lambda z: np.interp(z, redshifts, distances),
    )
    redshift_axis = radial_axis.secondary_yaxis("right", functions=conversions)
    ticks = z_step * np.arange(1, round(z_max / z_step))
    redshift_axis.set_yticks(ticks[1::2])
    redshift_axis.set_yticks(ticks[::2], minor=True)
    redshift_axis.tick_params(which="both", direction="out")
    redshift_axis.set_ylabel("redshift $z$")

## 1. Equatorial slice

The galaxy distribution is three-dimensional; a map on paper has two dimensions. Plotting every galaxy by right ascension and redshift, whatever its declination, stacks structures at different declinations on top of each other until they wash out. A strip a few degrees wide keeps them apart.

The sample is the **Main Galaxy Sample** of SDSS-I/II: galaxies brighter than Petrosian magnitude $r = 17.77$ in the imaging, targeted for a spectrum. These spectra carry the `GALAXY` bit in the `legacy_target1` column of `SpecObj`. The function `dbo.fPrimTarget('GALAXY')` returns that bit ($64$), so the condition names the flag instead of a magic number. Selecting by the target bit applies one selection rule across the whole strip. A `TOP n` without `ORDER BY` would instead return whichever $n$ rows the server happens to read first, which depends on how the table is stored.

In [ ]:
slice_limits = settings.parameter_set("equatorial_slice")
slice_limits

In [ ]:
slice_query = f"""
SELECT
    s.ra, s.dec, s.z,
    p.dered_g, p.dered_r,
    pz.kcorrG, pz.kcorrR
FROM SpecObj AS s
    JOIN PhotoObj AS p ON p.objID = s.bestObjID
    LEFT JOIN Photoz AS pz ON pz.objID = s.bestObjID
WHERE
    (s.legacy_target1 & dbo.fPrimTarget('GALAXY')) > 0
    AND s.class = 'GALAXY'
    AND s.zWarning = 0
    AND s.z BETWEEN {slice_limits["z_min"]} AND {slice_limits["z_max"]}
    AND s.dec BETWEEN {slice_limits["dec_min"]} AND {slice_limits["dec_max"]}
"""

The `JOIN` to `PhotoObj` brings the photometry of the object each spectrum belongs to (`bestObjID`). The `LEFT JOIN` to `Photoz` brings the k-corrections used below. A plain `JOIN` would silently drop the galaxies that have no `Photoz` row and punch holes in the map; with `LEFT JOIN` they stay, with `NULL` k-corrections.

In [ ]:
%%time
galaxies = CasJobs.executeQuery(
    sql=slice_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
print(f"{len(galaxies):,} galaxies")

### Rest-frame color

A galaxy's observed $g - r$ color depends on its redshift as well as on its stars: the farther the galaxy, the bluer the part of its spectrum that is shifted into the $g$ and $r$ filters. `Photoz` is the table of photometric redshift estimates; alongside the estimate it stores **k-corrections**, which convert each magnitude to the value it would have at $z = 0$. Subtracting them gives the rest-frame color. The k-corrections belong to the photometric redshift, not the spectroscopic one; for these galaxies the two typically differ by less than $0.02$, which shifts $g - r$ by about $0.02$.

The left panel shows the observed color: the red galaxies form a sequence that climbs by about $0.3$ across the slice. On a fixed color scale this would paint the outer part of the map red whatever the galaxies are. The right panel shows the rest-frame color, where the red sequence stays flat and separates from the blue, star-forming galaxies below it.

In [ ]:
galaxies["observed_gr"] = galaxies["dered_g"] - galaxies["dered_r"]
k_correction_gr = galaxies["kcorrG"] - galaxies["kcorrR"]
galaxies["rest_gr"] = galaxies["observed_gr"] - k_correction_gr
distance = settings.COSMOLOGY.comoving_distance(galaxies["z"].to_numpy())
galaxies["distance"] = distance.value
print(f"{galaxies['rest_gr'].isna().sum()} galaxies without k-correction")

In [ ]:
corrected = galaxies.dropna(subset=["rest_gr"])
color_bins = np.linspace(0, 1.4, 71)
panels = [
    ("observed_gr", "observed color $g - r$"),
    ("rest_gr", "rest-frame color $g - r$"),
]
with plt.style.context(settings.LIGHT_STYLE):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(settings.FIGURE_WIDTH, 0.4 * settings.FIGURE_WIDTH),
        sharey=True,
    )
    for ax, (column, title) in zip(axes, panels):
        ax.hist2d(
            corrected["z"],
            corrected[column],
            bins=[60, color_bins],
            cmap="Greys",
            cmin=1,
        )
        ax.set_xlabel("redshift $z$")
        ax.set_title(title)
    axes[0].set_ylabel("color $g - r$")
    plt.show()

### The map

Each dot is one galaxy, placed by right ascension (angle) and comoving distance (radius), computed from its redshift with the Planck 2018 cosmology. The dots are tiny and partly transparent, so brightness builds up where galaxies crowd. They are drawn in random order so that neither color systematically covers the other.

The color scale runs from blue to red through light grey at $g - r = 0.6$, the gap between the two galaxy populations. Both ends are bright so that they stand out on the black background. Galaxies without a k-correction are drawn mid-grey.

In [ ]:
gr_cmap = LinearSegmentedColormap.from_list(
    "blue_grey_red", ["#3D8BFF", "#D9D9D9", "#FF4D2E"]
).with_extremes(bad="0.5")
gr_norm = Normalize(vmin=0.35, vmax=0.85)
shuffled = galaxies.sample(frac=1, random_state=0)

with plt.style.context(settings.DARK_STYLE):
    fig, ax = plt.subplots(
        figsize=(settings.FIGURE_WIDTH, 1.15 * settings.FIGURE_WIDTH),
        subplot_kw={"projection": "polar"},
    )
    ax.set_theta_offset(np.deg2rad(-10))  # RA 190 deg points left
    points = ax.scatter(
        np.deg2rad(shuffled["ra"]),
        shuffled["distance"],
        c=shuffled["rest_gr"],
        cmap=gr_cmap,
        norm=gr_norm,
        s=0.3,
        alpha=0.7,
        linewidths=0,
        rasterized=True,
    )
    draw_map_axes(ax, slice_limits["z_max"], 0.01)
    colorbar = fig.colorbar(
        points, ax=ax, orientation="horizontal", shrink=0.4, pad=0.04
    )
    colorbar.set_label("rest-frame color $g - r$")
    colorbar.solids.set_alpha(1)
    plt.show()

What to look for:

- **Walls, filaments and voids.** Galaxies line up in thin walls and filaments around nearly empty voids tens of $\mathrm{Mpc}$ across. The long, dense wall about $350\,\mathrm{Mpc}$ away in the left fan ($z \approx 0.08$) is the Sloan Great Wall, which Gott et al. (2005) found in early SDSS data.
- **Red galaxies in clusters.** Red, mostly elliptical galaxies concentrate in the densest knots; blue star-forming galaxies fill the filaments and the edges of voids.
- **Fingers of god.** Clusters appear as short spikes pointing at the observer, such as Abell 119 and Abell 168 about $190\,\mathrm{Mpc}$ away in the right fan. The map uses redshift as distance, and the orbital velocities of the galaxies inside a cluster, up to about $1000\,\mathrm{km\,s^{-1}}$, add Doppler shifts that stretch the cluster along the line of sight.
- **Thinning and reddening with distance.** The sample is flux-limited: far away only the most luminous galaxies pass the $r = 17.77$ cut. The density of dots therefore falls outward, although the Universe does not thin out, and the outer parts look redder, because the most luminous galaxies are mostly red ellipticals.
- **Fan edges.** The left fan is the northern SDSS imaging area, the right fan is Stripe 82 in the southern Galactic sky. The empty angles between them point towards the plane of the Milky Way, where the survey did not observe.
- **Fan thickness.** Stripe 82 is only $2.5^\circ$ wide (declination $-1.25^\circ$ to $+1.25^\circ$), so the right fan is a thinner slice than the left one and holds fewer galaxies per area of the map.

## 2. Multi-tracer wedge

Each SDSS spectroscopic survey targeted a different kind of object, chosen to reach a different distance: the bright galaxies of the Main Galaxy Sample (SDSS-I/II) out to $z \approx 0.25$, the luminous red galaxies of BOSS (SDSS-III) out to $z \approx 0.8$, the luminous red and emission-line galaxies of eBOSS (SDSS-IV) out to $z \approx 1.1$, and quasars out to $z \approx 3.5$ and beyond. Cosmologists call such a sample a *tracer* of the matter distribution.

`tracers.json`, next to this notebook, defines each tracer by an SQL condition, a redshift range, and a plot color and dot size. In `SpecObj`, `survey` names the survey a spectrum belongs to, `sourceType` the class it was targeted as, and `class` what the spectrum turned out to be. Adding a tracer means adding an entry to the file; the code below stays the same.

In [ ]:
wedge_limits = settings.parameter_set("multi_tracer_wedge")
tracers = pd.DataFrame(settings.tracers()).set_index("name")
tracers

In [ ]:
def tracer_query(tracer, columns, limits):
    """SQL query selecting one tracer inside a declination strip."""
    return f"""
SELECT {columns}
FROM SpecObj AS s
WHERE
    {tracer["where"]}
    AND s.zWarning = 0
    AND s.z BETWEEN {tracer["z_min"]} AND {tracer["z_max"]}
    AND s.dec BETWEEN {limits["dec_min"]} AND {limits["dec_max"]}
"""


print(tracer_query(tracers.loc["Quasars"], "s.ra, s.z", wedge_limits))

Before downloading, count each tracer on the server. `COUNT(*)` reads the same rows as the full query but sends back a single number, so it tells how large the download will be at almost no transfer cost.

In [ ]:
counts = {}
for name, tracer in tracers.iterrows():
    query = tracer_query(tracer, "COUNT(*) AS n", wedge_limits)
    result = CasJobs.executeQuery(
        sql=query, context=settings.CASJOBS_CONTEXT, format="pandas"
    )
    counts[name] = result["n"].iloc[0]
pd.Series(counts, name="rows")

In [ ]:
%%time
samples = []
for name, tracer in tracers.iterrows():
    query = tracer_query(tracer, "s.ra, s.z", wedge_limits)
    sample = CasJobs.executeQuery(
        sql=query, context=settings.CASJOBS_CONTEXT, format="pandas"
    )
    sample["tracer"] = name
    samples.append(sample)
wedge = pd.concat(samples, ignore_index=True)
distance = settings.COSMOLOGY.comoving_distance(wedge["z"].to_numpy())
wedge["distance"] = distance.value
print(f"{len(wedge):,} objects")

### The map

Colors mark the tracers; the radius is comoving distance as before, so the whole first map now fits in the small disk at the center. Sparse tracers get larger dots so that they stay visible next to the dense ones.

In [ ]:
shuffled = wedge.sample(frac=1, random_state=0)
colors = shuffled["tracer"].map(tracers["color"])
sizes = shuffled["tracer"].map(tracers["size"])
legend_handles = [
    Line2D([], [], linestyle="", marker="o", color=color, label=name)
    for name, color in tracers["color"].items()
]

with plt.style.context(settings.DARK_STYLE):
    fig, ax = plt.subplots(
        figsize=(settings.FIGURE_WIDTH, 1.1 * settings.FIGURE_WIDTH),
        subplot_kw={"projection": "polar"},
    )
    ax.set_theta_offset(np.deg2rad(-10))  # RA 190 deg points left
    ax.scatter(
        np.deg2rad(shuffled["ra"]),
        shuffled["distance"],
        c=colors,
        s=sizes,
        alpha=0.6,
        linewidths=0,
        rasterized=True,
    )
    draw_map_axes(ax, tracers["z_max"].max(), 0.25)
    fig.legend(handles=legend_handles, loc="outside lower center", ncols=3)
    plt.show()

What to look for:

- **Shells of tracers.** Each tracer fills the range of distances its survey was designed for. Where one tracer hands over to the next, the color and density of dots change, but the Universe does not.
- **Fading structure.** The galaxy samples still show the clumpy cosmic web out to about $3\,\mathrm{Gpc}$; the sparse quasars beyond only hint at it.
- **Survey footprints.** eBOSS observed emission-line galaxies only in a few small areas of sky; in this strip they appear in the right fan alone. The right fan also holds about $3.5$ times as many quasars per degree of right ascension as the left one: Stripe 82 was imaged many times, so its quasars could also be selected by their variability, and several extra programmes targeted it.
- **Single plates.** The thin spokes in the empty sector at the top are quasars on individual spectroscopic plates of special programmes outside the main survey, such as the SEGUE plates for Milky Way stars and the BOSS plates towards Orion and Taurus near RA $80$–$90^\circ$. Each plate covers a circle $3^\circ$ across, so it shows up as a narrow wedge.
- **Look-back time.** The quasars at $z = 3.5$ are seen as they were $12$ billion years ago, so the outer part of the map shows an earlier Universe as well as a more distant one.